# Lesson 02: Optimization, Regression, and Classification - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: preprocessing, regression, logistic classification, metrics, and cross-validation.

## Learning goals
1. Clean missing values and clipped sensor outliers.
2. Fit a regression model for heat duty.
3. Fit a classifier for unsafe operation.
4. Report recall, precision, F1, and cross-validation performance.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score

rng = np.random.default_rng(12)
n = 160
T = rng.normal(350, 18, n)
P = rng.normal(8, 1.2, n)
flow = rng.normal(2.0, 0.35, n)
heat_duty = 0.8*T + 12*P - 35*flow + rng.normal(0, 8, n)
unsafe = ((T > 372) | (P > 9.4) | ((T > 360) & (flow < 1.7))).astype(int)

X = pd.DataFrame({"T": T, "P": P, "flow": flow})
X.loc[rng.choice(n, 8, replace=False), "T"] = np.nan
X.loc[rng.choice(n, 3, replace=False), "P"] = 9999
X.loc[X["P"] > 100, "P"] = np.nan

reg = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression())
clf = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))

X_train, X_test, y_train, y_test = train_test_split(X, heat_duty, test_size=0.25, random_state=0)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)
print(f"Regression RMSE: {mean_squared_error(y_test, pred, squared=False):.2f}")

X_train, X_test, y_train, y_test = train_test_split(X, unsafe, test_size=0.25, stratify=unsafe, random_state=0)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

cv_recall = cross_val_score(clf, X, unsafe, cv=5, scoring="recall")
print(f"5-fold recall: {cv_recall.mean():.3f} +/- {cv_recall.std():.3f}")
print("\nSafety interpretation: a false negative is a missed unsafe state, so recall is usually the priority metric.")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?